# Course Engagement Analysis

## Project Overview
Analyze LMS (Learning Management System) activity data to measure student participation, content usage, assignment behavior, and likely engagement drivers.

## Learning Objectives
- Quantify student engagement through LMS activity metrics
- Identify patterns in content consumption and assignment submission
- Segment students by engagement level
- Correlate engagement metrics with course outcomes

## Problem Statement
Course designers and instructors need to understand how students interact with course materials to identify disengaged learners early and improve content effectiveness.

## Why This Project Matters
Engagement is the strongest leading indicator of course success. Identifying low-engagement patterns early enables timely interventions that improve completion and learning outcomes.

## Dataset Overview
Synthetic LMS dataset: ~3,000 student-course enrollments with login frequency, video views, assignment submissions, forum posts, quiz scores, and final grade.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 3000
# Base engagement level drives all metrics
engagement_base = np.clip(np.random.beta(2, 2, n), 0, 1)

logins = np.clip((engagement_base * 80 + np.random.normal(0, 10, n)).astype(int), 0, 120)
video_views = np.clip((engagement_base * 50 + np.random.normal(0, 8, n)).astype(int), 0, 80)
assignments_submitted = np.clip((engagement_base * 10 + np.random.normal(0, 1.5, n)).astype(int), 0, 12)
assignments_total = 12
forum_posts = np.clip((engagement_base * 15 + np.random.exponential(2, n)).astype(int), 0, 40)
quiz_avg = np.clip(engagement_base * 60 + 30 + np.random.normal(0, 8, n), 0, 100).round(1)
time_spent_hrs = np.clip(engagement_base * 80 + np.random.normal(0, 12, n), 1, 120).round(1)
final_grade = np.clip(engagement_base * 55 + 35 + np.random.normal(0, 7, n), 0, 100).round(1)

course = np.random.choice(['Intro CS', 'Statistics', 'Marketing', 'Biology', 'Psychology'], n, p=[0.22, 0.20, 0.18, 0.22, 0.18])
term = np.random.choice(['Fall 2023', 'Spring 2024'], n, p=[0.52, 0.48])
modality = np.random.choice(['Online', 'Hybrid', 'In-Person'], n, p=[0.40, 0.30, 0.30])

df = pd.DataFrame({
    'StudentID': [f'S{i:05d}' for i in range(n)],
    'Course': course, 'Term': term, 'Modality': modality,
    'Logins': logins, 'VideoViews': video_views,
    'AssignmentsSubmitted': assignments_submitted, 'AssignmentsTotal': assignments_total,
    'ForumPosts': forum_posts, 'QuizAvg': quiz_avg,
    'TimeSpentHrs': time_spent_hrs, 'FinalGrade': final_grade
})
df['SubmissionRate'] = (df['AssignmentsSubmitted'] / df['AssignmentsTotal']).round(3)
df['Passed'] = df['FinalGrade'] >= 60
print(f'Dataset: {df.shape}')
print(f'Pass rate: {df["Passed"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nEngagement stats:')
for col in ['Logins', 'VideoViews', 'ForumPosts', 'TimeSpentHrs']:
    print(f'  {col}: mean={df[col].mean():.1f}, median={df[col].median():.1f}')

## Engagement Metric Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col, color in zip(axes.flat, ['Logins', 'VideoViews', 'TimeSpentHrs', 'ForumPosts'],
                            ['steelblue', 'coral', 'teal', 'orange']):
    df[col].hist(bins=30, ax=ax, edgecolor='black', color=color, alpha=0.8)
    ax.set_title(f'Distribution of {col}')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.1f}')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'engagement_distributions.png'), dpi=100, bbox_inches='tight')
plt.show()

## Engagement vs Final Grade

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flat, ['Logins', 'VideoViews', 'TimeSpentHrs', 'SubmissionRate']):
    ax.scatter(df[col], df['FinalGrade'], alpha=0.15, s=10)
    z = np.polyfit(df[col], df['FinalGrade'], 1)
    ax.plot(np.sort(df[col]), np.polyval(z, np.sort(df[col])), 'r-', lw=2)
    r = df[col].corr(df['FinalGrade'])
    ax.set_title(f'{col} vs Final Grade (r={r:.2f})')
    ax.set_xlabel(col)
    ax.set_ylabel('Final Grade')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'engagement_vs_grade.png'), dpi=100, bbox_inches='tight')
plt.show()

## Correlation Matrix

In [ ]:
corr_cols = ['Logins', 'VideoViews', 'ForumPosts', 'TimeSpentHrs', 'SubmissionRate', 'QuizAvg', 'FinalGrade']
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Engagement Metric Correlations')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlation_matrix.png'), dpi=100, bbox_inches='tight')
plt.show()

## Engagement by Modality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mod_engagement = df.groupby('Modality')[['Logins', 'VideoViews', 'ForumPosts']].mean()
mod_engagement.plot.bar(ax=axes[0], edgecolor='black')
axes[0].set_title('Avg Engagement Metrics by Modality')
axes[0].tick_params(axis='x', rotation=0)
mod_grade = df.groupby('Modality')['FinalGrade'].mean()
mod_grade.plot.bar(ax=axes[1], edgecolor='black', color='teal')
axes[1].set_title('Avg Final Grade by Modality')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'modality_engagement.png'), dpi=100, bbox_inches='tight')
plt.show()

## Engagement Segmentation

In [ ]:
# Create engagement tiers based on composite score
df['EngagementScore'] = (
    df['Logins'] / df['Logins'].max() * 0.25 +
    df['VideoViews'] / df['VideoViews'].max() * 0.25 +
    df['SubmissionRate'] * 0.30 +
    df['ForumPosts'] / df['ForumPosts'].max() * 0.20
).round(3)
df['EngagementTier'] = pd.cut(df['EngagementScore'], bins=[0, 0.3, 0.55, 0.75, 1.0],
                                labels=['Low', 'Medium', 'High', 'Very High'])
tier_stats = df.groupby('EngagementTier', observed=True).agg(
    Count=('StudentID', 'count'),
    AvgGrade=('FinalGrade', 'mean'),
    PassRate=('Passed', 'mean')
).round(2)
print('Engagement Tier Summary:')
print(tier_stats)

fig, ax = plt.subplots(figsize=(10, 5))
tier_stats[['AvgGrade']].plot.bar(ax=ax, edgecolor='black', color='coral')
ax.set_title('Average Grade by Engagement Tier')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'engagement_tiers.png'), dpi=100, bbox_inches='tight')
plt.show()

## Assignment Submission Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['AssignmentsSubmitted'].value_counts().sort_index().plot.bar(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Distribution of Assignments Submitted (out of 12)')
pass_by_sub = df.groupby('AssignmentsSubmitted')['Passed'].mean()
pass_by_sub.plot(ax=axes[1], marker='o', color='coral')
axes[1].set_title('Pass Rate by Assignments Submitted')
axes[1].set_xlabel('Assignments Submitted')
axes[1].set_ylabel('Pass Rate')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'assignment_patterns.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Assignment submission rate** is the strongest single predictor of course success
- **All engagement metrics** are positively correlated with final grade — the relationship is robust
- **Low-engagement tier** students have dramatically lower pass rates — they are intervention candidates
- **Modality** shows modest differences — online students may need more proactive engagement nudges
- **Forum participation** has the weakest individual correlation but still contributes to the engagement composite
- Early identification of low-engagement students (weeks 2-3) could enable timely interventions

## Limitations
- Synthetic data with simplified engagement model
- No temporal granularity (weekly patterns)
- No content-level analysis (which materials most viewed)
- No student demographics or prior performance
- No A/B testing of interventions

## How to Improve This Project
- Add weekly engagement trajectories for early warning
- Include content-level analytics (which videos, which pages)
- Build predictive models for at-risk student detection
- Add intervention tracking (nudge emails, office hours)
- Compare engagement across semesters for trend analysis

## Production Considerations
- Real-time LMS dashboards for instructors
- Automated early-warning alerts for disengaged students
- Personalized engagement nudges via email/notification
- Course design feedback loops based on content usage data

## Common Mistakes
- Equating login count with genuine engagement
- Not controlling for course difficulty when comparing grades
- Treating engagement as a single metric instead of multidimensional
- Ignoring that some high-performing students engage minimally (already proficient)

## Mini Challenge / Exercises
1. What engagement metric has the highest correlation with final grade?
2. What is the minimum number of assignments submitted where pass rate exceeds 80%?
3. Create a new engagement metric using only logins and video views. How well does it predict grades?
4. Compare the engagement tier distribution across courses — which course has the most low-engagement students?

## Final Summary / Key Takeaways
- Student engagement is multidimensional — no single metric captures it fully
- Assignment submission is the most actionable engagement indicator
- Engagement tiers clearly separate successful from struggling students
- Early engagement data (first 2-3 weeks) can power effective early warning systems
- Data-driven course design improves both engagement and learning outcomes